# Task3 : Build a Chatbot using Hugging Face Transformers

Step-1:Install the requirements

In [1]:
!pip install -q -U "transformers>=4.37.0" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 35.6 MB/s eta 0:00:00


Step-2:Importing the packages

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

Step-3:Loading the model

Here we are using Qwen which is a lightweight instruction-tuned language model from Alibaba under the Qwen (Tongyi) series.

It has 0.5 billion parameter (0.5B) transformer model,designed for instruction-following tasks (chat, Q&A, basic reasoning) and it is part of the newer Qwen2.5 family

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
print("Model loaded successfully!\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!



Step-4: Defining response function

the response function Takes user input + history and Returns model response + updated history.

Here,the input is take, then adds it to the history, generates a response and also adds that to the history

In [4]:
def generate_response(user_input, chat_history):
    #Add user message to the history
    chat_history.append({"role": "user", "content": user_input})
    #Formats the conversation using chat template
    input_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    #it generates the response
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    #decodes only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    )
    #Adds the bot response to history
    chat_history.append({"role": "assistant", "content": response})
    return response, chat_history

Step-5:defining loop for chatbot

this fucntion keeps the bot running until we dont type exit or quit.

In [5]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    chat_history = []

    while True:
        user_input = input("You: ")

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day 👋")
            break

        # Generate response
        reply, chat_history = generate_response(user_input, chat_history)

        print(f"Chatbot: {reply}\n")


Chatbot Workflow:

User Input → Formatting → Tokenization → Model Generation → Response Decoding → Display Output → Repeat Until Exit

In [6]:
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans do. It involves the development of algorithms, models, and systems designed to perform tasks that would typically require human intelligence.

Key characteristics of AI include:
1. **Learning**: The ability of machines to improve their performance over time through experience.
2. **Self-organization**: Machines can operate without strict instructions or pre-programmed rules.
3. **Intelligence**: The capability to understand language, recognize patterns, solve problems, make decisions, and adapt to new situations.
4. **Problem-solving**: The ability to tackle complex problems and find solutions efficiently.
5. **Natural language processing (NLP)**: Ability to understand and

You: Who created python?
Chatbot: Python was created by Guido va